# 1. Install Liberaries

In [4]:
!pip install numpy pandas matplotlib seaborn scikit-learn xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.8/101.7 MB 3.7 MB/s eta 0:00:27
    --------------------------------------- 1.8/101.7 MB 3.4 MB/s eta 0:00:30
   - -------------------------------------- 2.6/101.7 MB 3.5 MB/s eta 0:00:29
   - -------------------------------------- 3.1/101.7 MB 3.5 MB/s eta 0:00:29
   - -------------------------------------- 3.9/101.7 MB 3.5 MB/s eta 0:00:29
   - -------------------------------------- 4.7/101.7 MB 3.4 MB/s eta 0:00:29
   -- ------------------------------------- 5.2/101.7 MB 3.4 MB/s eta 0:00:29
   -- ------------------------------------- 6.0/101.7 MB 3.5 MB/s eta 0:00:28
   -- ------------------------------------- 6.8/101.7 MB 3.5 MB/s eta 0:00:28
   -- ------------------------------------- 7.6/101.7 MB 3.5 MB/s eta 0:00:28
   --- ------------------------------------ 8.4/101.7 MB 3.5 MB/s eta 0:00:27


# 2. Data loading and visualizations

In [3]:
import os
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import time

warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)

from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier


# =========================================================
# PATHS
# =========================================================

DATA_DIR = r"C:\Users\emade\Downloads\Dementia_Detection\data"
RESULTS_DIR = r"C:\Users\emade\Downloads\Dementia_Detection\results\visuals"

os.makedirs(RESULTS_DIR, exist_ok=True)

DEMO_PATH = os.path.join(DATA_DIR, "oasis_longitudinal_demographics.xlsx")
PRED_PATH = os.path.join(DATA_DIR, "Predictions.xlsx")


# =========================================================
# VISUAL STYLE
# =========================================================

PALETTE = {
    "Nondemented": "#2196F3",
    "Demented": "#F44336",
    "Converted": "#FF9800"
}

sns.set_theme(style="whitegrid", font_scale=1.1)


# =========================================================
# LOAD DATA
# =========================================================

demo = pd.read_excel(DEMO_PATH)
pred = pd.read_excel(PRED_PATH)

print("✓ Data loaded successfully")


# =========================================================
# FIGURE 1 — EDA VISUALIZATION
# =========================================================

fig1 = plt.figure(figsize=(20, 16))

fig1.suptitle(
    "OASIS Longitudinal Dataset — Exploratory Data Analysis",
    fontsize=18,
    fontweight='bold',
    y=0.98
)

gs = gridspec.GridSpec(
    3,
    3,
    figure=fig1,
    hspace=0.45,
    wspace=0.38
)


# =========================================================
# 1. GROUP DISTRIBUTION
# =========================================================

ax = fig1.add_subplot(gs[0, 0])

grp_counts = demo['Group'].value_counts()

colors = [PALETTE[g] for g in grp_counts.index]

ax.pie(
    grp_counts,
    labels=grp_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=140,
    textprops={'fontsize': 10}
)

ax.set_title(
    "Group Distribution\n(373 MRI sessions)",
    fontweight='bold'
)


# =========================================================
# 2. SEX DISTRIBUTION
# =========================================================

ax = fig1.add_subplot(gs[0, 1])

sex_grp = demo.groupby(['Group', 'M/F']).size().unstack()

sex_grp.plot(
    kind='bar',
    ax=ax,
    color=['#E91E63', '#42A5F5'],
    edgecolor='white',
    width=0.6
)

ax.set_title(
    "Sex Distribution by Group",
    fontweight='bold'
)

ax.set_xlabel("Group")
ax.set_ylabel("Count")

ax.tick_params(axis='x', rotation=15)

ax.legend(title='Sex')


# =========================================================
# 3. MMSE DISTRIBUTION
# =========================================================

ax = fig1.add_subplot(gs[1, 0])

for grp, col in PALETTE.items():

    subset = demo[demo['Group'] == grp]['MMSE'].dropna()

    ax.hist(
        subset,
        bins=10,
        alpha=0.65,
        color=col,
        label=grp,
        edgecolor='white'
    )

ax.set_title(
    "MMSE Score Distribution",
    fontweight='bold'
)

ax.set_xlabel("MMSE Score")
ax.set_ylabel("Count")

ax.legend()


# =========================================================
# 4. CDR BOXPLOT
# =========================================================

ax = fig1.add_subplot(gs[1, 1])

grp_order = ['Nondemented', 'Converted', 'Demented']

demo_plot = demo[demo['Group'].isin(grp_order)]

sns.boxplot(
    data=demo_plot,
    x='Group',
    y='CDR',
    palette=PALETTE,
    order=grp_order,
    ax=ax,
    width=0.5
)

ax.set_title(
    "CDR Score by Group",
    fontweight='bold'
)

ax.set_xlabel("")
ax.set_ylabel("CDR")

ax.tick_params(axis='x', rotation=10)


# =========================================================
# 5. VISITS PER SUBJECT
# =========================================================

ax = fig1.add_subplot(gs[2, 0])

visits_per_subj = demo.groupby('Subject ID')['Visit'].max()

ax.hist(
    visits_per_subj,
    bins=range(1, 7),
    align='left',
    color='#7B1FA2',
    edgecolor='white',
    rwidth=0.75
)

ax.set_title(
    "Visits per Subject",
    fontweight='bold'
)

ax.set_xlabel("Number of Visits")
ax.set_ylabel("Subjects")

ax.set_xticks(range(1, 6))


# =========================================================
# 6. EDUCATION LEVEL
# =========================================================

ax = fig1.add_subplot(gs[2, 1])

sns.boxplot(
    data=demo_plot,
    x='Group',
    y='EDUC',
    palette=PALETTE,
    order=grp_order,
    ax=ax,
    width=0.5
)

ax.set_title(
    "Education Level by Group",
    fontweight='bold'
)

ax.set_xlabel("")
ax.set_ylabel("Years of Education")

ax.tick_params(axis='x', rotation=10)


# =========================================================
# 7. CORRELATION HEATMAP
# =========================================================

ax = fig1.add_subplot(gs[2, 2])

num_cols = [
    'Age',
    'EDUC',
    'SES',
    'MMSE',
    'CDR',
    'eTIV',
    'nWBV',
    'ASF'
]

corr = demo[num_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    ax=ax,
    annot_kws={'size': 8},
    linewidths=0.4
)

ax.set_title(
    "Feature Correlation Matrix",
    fontweight='bold'
)

ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', rotation=0, labelsize=8)


# =========================================================
# SAVE FIGURE
# =========================================================

save_path = os.path.join(RESULTS_DIR, "fig1_eda.png")

plt.savefig(
    save_path,
    dpi=300,
    bbox_inches='tight'
)
plt.show()
plt.close(fig1)

print(f"✓ Figure saved successfully:\n{save_path}")

✓ Data loaded successfully
✓ Figure saved successfully:
C:\Users\emade\Downloads\Dementia_Detection\results\visuals\fig1_eda.png


# 3. Data Encoding & Preprocessing

In [4]:

df = demo.copy()

# Fill nulls with mean
for col in ['SES', 'MMSE']:
    df[col] = df[col].fillna(df[col].mean())

# Encode categoricals
le = LabelEncoder()
df['Group_enc'] = le.fit_transform(df['Group'])      # Demented=0, Nondemented=1, Converted=2
df['Sex_enc']   = (df['M/F'] == 'M').astype(int)

# Features & target
FEATURES = ['Age', 'Sex_enc', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF', 'MR Delay']
X = df[FEATURES].values
y = df['Group_enc'].values

# ─────────────────────────────────────────────
# PREPROCESSING FIGURE — Figure 2
# ─────────────────────────────────────────────
fig2, axes = plt.subplots(2, 3, figsize=(18, 10))
fig2.suptitle("Preprocessing Insights", fontsize=16, fontweight='bold')

# Class balance
ax = axes[0, 0]
class_names = le.classes_
counts = [np.sum(y == i) for i in range(len(class_names))]
bars = ax.bar(class_names, counts, color=[PALETTE[c] for c in class_names], edgecolor='white', width=0.55)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5, str(cnt),
            ha='center', fontsize=11, fontweight='bold')
ax.set_title("Class Balance (Post-Imputation)", fontweight='bold')
ax.set_ylabel("Count")

# Outlier detection via IQR - boxplots after scaling
ax = axes[0, 1]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
feature_labels = ['Age', 'Sex', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF', 'MR Delay']
bp = ax.boxplot(X_scaled, labels=feature_labels, patch_artist=True,
                boxprops=dict(facecolor='#90CAF9', color='#1565C0'),
                medianprops=dict(color='#B71C1C', linewidth=2))
ax.set_title("Feature Distributions (Standardized)\n— Outlier View", fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.set_ylabel("Z-score")

# Null heatmap (before fill)
ax = axes[0, 2]
null_map = demo[['Age', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF', 'MR Delay']].isnull().astype(int)
if null_map.any().any():
    sns.heatmap(null_map.T, cbar=False, cmap='Reds', ax=ax, linewidths=0.0)
    ax.set_title("Missing Values (Red = Null)", fontweight='bold')
else:
    ax.text(0.5, 0.5, "No Missing Values\nAfter Imputation",
            ha='center', va='center', fontsize=13, transform=ax.transAxes)
    ax.set_title("Null Check", fontweight='bold')
    ax.axis('off')
ax.set_xlabel("Session Index")

# Duplicate check
ax = axes[1, 0]
dups = demo.duplicated().sum()
ax.bar(['Unique', 'Duplicate'], [len(demo) - dups, dups], color=['#4CAF50', '#F44336'], edgecolor='white')
ax.set_title(f"Duplicate Rows: {dups}", fontweight='bold')
ax.set_ylabel("Count")

# MMSE before/after imputation
ax = axes[1, 1]
missing_idx = demo['MMSE'].isnull()
ax.hist(demo['MMSE'].dropna(), bins=15, color='#42A5F5', alpha=0.7, label='Original', edgecolor='white')
ax.axvline(demo['MMSE'].mean(), color='red', linestyle='--', linewidth=2,
           label=f'Mean fill ({demo["MMSE"].mean():.1f})')
ax.scatter([demo['MMSE'].mean()] * missing_idx.sum(),
           [5] * missing_idx.sum(), color='red', zorder=5, s=60, label=f'{missing_idx.sum()} filled')
ax.set_title("MMSE: Null Filled with Mean", fontweight='bold')
ax.set_xlabel("MMSE"); ax.legend(fontsize=9)

# SES distribution
ax = axes[1, 2]
ax.hist(demo['SES'].dropna(), bins=8, color='#AB47BC', alpha=0.7, edgecolor='white')
ax.axvline(demo['SES'].mean(), color='red', linestyle='--', linewidth=2,
           label=f'Mean fill ({demo["SES"].mean():.2f})')
ax.set_title(f"SES: 19 Nulls Filled with Mean", fontweight='bold')
ax.set_xlabel("SES Score"); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(r'C:\Users\emade\Downloads\fig2_preprocessing.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()
print("✓ Figure 2 saved")


✓ Figure 2 saved


# 4. Save Processed Data

In [6]:
df.to_csv(r"C:\Users\emade\Downloads\Dementia_Detection\data\processed\processed_data.csv", index=False)

np.save(r"C:\Users\emade\Downloads\Dementia_Detection\data\processed\X_scaled.npy", X_scaled)
np.save(r"C:\Users\emade\Downloads\Dementia_Detection\data\processed\y.npy", y)

In [8]:
import joblib

joblib.dump(scaler, r"C:\Users\emade\Downloads\Dementia_Detection\data\processed\scaler.pkl")
joblib.dump(le, r"C:\Users\emade\Downloads\Dementia_Detection\data\processed\label_encoder.pkl")

['C:\\Users\\emade\\Downloads\\Dementia_Detection\\data\\processed\\label_encoder.pkl']